# Cosm synthetic data foundation
This notebook generates realistic synthetic data for **Cosm**, the immersive "Shared Reality" dome entertainment company, and lands it in Unity Catalog `jr_febar_catalog`. It also ingests public data with **Lakeflow Connect** (Auto Loader file connector).

| Schema | Tables | What it represents |
| --- | --- | --- |
| `venues` | `venues`, `seating_zones` | Dome venues (LA, Dallas open; Atlanta, Detroit in development) and seating tiers |
| `content` | `content_catalog`, `showtimes` | Live sports, immersive films, concerts, special events and the daily show schedule |
| `customers` | `customers`, `memberships` | CRM profiles, rewards tier, Cosm Club subscriptions |
| `sales` | `ticket_orders`, `fnb_menu_items`, `fnb_transactions`, `fnb_transaction_items` | Ticketing and in-venue food and beverage POS |
| `marketing` | `campaigns`, `nps_survey_responses` | Campaigns / promo codes and post-visit surveys |
| `external_data` | `weather_daily_raw`, `weather_daily`, `us_public_holidays` | Public data: Open-Meteo historical weather + Nager.Date US holidays |

Data is internally consistent: orders never exceed showtime capacity, purchases occur before showtimes, members get discounts only while active, F&B only for attended orders. Data is "as of" 2026-11-19 with future showtimes on presale. Re-running is idempotent (tables are overwritten, seed fixed).

In [0]:
import json, os, requests
import numpy as np
import pandas as pd
from datetime import date, datetime
from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOG = "jr_febar_catalog"
SCHEMAS = {
    "venues":        "Cosm dome venues and seating zones",
    "content":       "Content catalog (Shared Reality live sports, immersive films, concerts, special events) and showtime schedule",
    "customers":     "Customer profiles (CRM) and Cosm Club memberships",
    "sales":         "Ticket orders and food and beverage point-of-sale transactions",
    "marketing":     "Marketing campaigns, promo codes and post-visit NPS surveys",
    "external_data": "Public third-party data ingested with Lakeflow Connect (Auto Loader file connector)",
}
for s, c in SCHEMAS.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{s} COMMENT '{c}'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.external_data.landing COMMENT 'Landing zone for raw public data files'")

SEED = 42
rng = np.random.default_rng(SEED)
NOW = pd.Timestamp("2026-11-19 09:00:00")   # 'as of' timestamp for the synthetic data
SCHEDULE_END = pd.Timestamp("2026-12-31")   # showtimes are scheduled through year end
LAUNCH = pd.Timestamp("2024-03-01")         # pre-opening waitlist launch

def _spark_type(series):
    k = series.dtype.kind
    if k in "iu": return T.LongType()
    if k == "f":  return T.DoubleType()
    if k == "b":  return T.BooleanType()
    if k == "M":  return T.TimestampType()
    nn = series.dropna()
    v = nn.iloc[0] if len(nn) else ""
    if isinstance(v, datetime): return T.TimestampType()
    if isinstance(v, date):     return T.DateType()
    if isinstance(v, (bool, np.bool_)): return T.BooleanType()
    return T.StringType()

def save(pdf, schema, table, comment, chunk=200_000):
    """Write a pandas DataFrame to a UC Delta table in chunks with an explicit schema."""
    fq = f"{CATALOG}.{schema}.{table}"
    pdf = pdf.reset_index(drop=True)
    sch = T.StructType([T.StructField(c, _spark_type(pdf[c]), True) for c in pdf.columns])
    for i, start in enumerate(range(0, len(pdf), chunk)):
        w = spark.createDataFrame(pdf.iloc[start:start + chunk], schema=sch).write
        w = w.mode("overwrite").option("overwriteSchema", "true") if i == 0 else w.mode("append")
        w.saveAsTable(fq)
    safe = comment.replace("'", "''")
    spark.sql(f"COMMENT ON TABLE {fq} IS '{safe}'")
    print(f"{fq}: {spark.table(fq).count():,} rows")

In [0]:
venues_pdf = pd.DataFrame([
    dict(venue_id="COSM-LA",  venue_name="Cosm Los Angeles", city="Inglewood",  state="CA", market="Los Angeles",       district="Hollywood Park",    latitude=33.9535, longitude=-118.3392, timezone="America/Los_Angeles", dome_diameter_ft=87, seating_capacity=1500, open_date=date(2024, 6, 28), status="Open"),
    dict(venue_id="COSM-DAL", venue_name="Cosm Dallas",      city="The Colony", state="TX", market="Dallas-Fort Worth", district="Grandscape",        latitude=33.0806, longitude=-96.8836,  timezone="America/Chicago",     dome_diameter_ft=87, seating_capacity=1400, open_date=date(2025, 6, 20), status="Open"),
    dict(venue_id="COSM-ATL", venue_name="Cosm Atlanta",     city="Atlanta",    state="GA", market="Atlanta",           district="Centennial Yards",  latitude=33.7550, longitude=-84.4000,  timezone="America/New_York",    dome_diameter_ft=87, seating_capacity=1500, open_date=None,              status="In Development"),
    dict(venue_id="COSM-DET", venue_name="Cosm Detroit",     city="Detroit",    state="MI", market="Detroit",           district="District Detroit",  latitude=42.3410, longitude=-83.0550,  timezone="America/Detroit",     dome_diameter_ft=87, seating_capacity=1450, open_date=None,              status="In Development"),
])

# code, name, share of capacity, price multiplier, description
ZONES = [
    ("GA",  "Dome Floor General Admission", 0.45, 1.00, "Tiered general admission seating closest to the dome"),
    ("CTR", "Center Tier",                  0.25, 1.40, "Reserved center seating with optimal sightlines"),
    ("UPR", "Upper Tier",                   0.15, 0.85, "Reserved upper seating with full dome view"),
    ("BTH", "Premium Booths",               0.10, 2.20, "Booth seating with in-seat food and beverage service"),
    ("STE", "Suites",                       0.05, 3.50, "Private suites with dedicated host and premium menu"),
]
zones_pdf = pd.DataFrame([
    dict(zone_id=f"{v.venue_id}-{code}", venue_id=v.venue_id, zone_code=code, zone_name=name,
         capacity=int(round(v.seating_capacity * share)), price_multiplier=mult, description=desc)
    for v in venues_pdf.itertuples() for code, name, share, mult, desc in ZONES
])

save(venues_pdf, "venues", "venues", "Cosm dome venues with location, capacity, open date and status")
save(zones_pdf, "venues", "seating_zones", "Seating zones per venue with capacity and price multiplier vs base ticket price")

In [0]:
ALL, WEEK = list(range(1, 13)), list(range(7))   # months; weekdays (Mon=0)
M, A, E, L = "MATINEE", "AFTERNOON", "EVENING", "LATE"
# id, title, category, genre, partner, months, weekdays, slots, base price, popularity, runtime min
CONTENT_SPEC = [
    ("SPT-NBA",  "NBA Shared Reality",                 "Live Sports",    "Basketball",    "NBA",            [10,11,12,1,2,3,4,5,6], WEEK,      [E, L],    45, 0.75, 150),
    ("SPT-NFL",  "NFL Game Day Dome",                  "Live Sports",    "Football",      "NFL",            [9,10,11,12,1,2],       [0,3,6],   [A, E],    55, 0.88, 195),
    ("SPT-CFB",  "College Football Saturdays",         "Live Sports",    "Football",      "NCAA",           [9,10,11,12,1],         [5],       [M, A, E], 40, 0.70, 210),
    ("SPT-NHL",  "NHL Ice Night",                      "Live Sports",    "Hockey",        "NHL",            [10,11,12,1,2,3,4,5,6], [1,2,3,4], [E],       35, 0.50, 160),
    ("SPT-MLB",  "MLB Under the Dome",                 "Live Sports",    "Baseball",      "MLB",            [3,4,5,6,7,8,9,10],     WEEK,      [A, E],    30, 0.45, 180),
    ("SPT-UFC",  "UFC Fight Night",                    "Live Sports",    "Combat Sports", "UFC",            ALL,                    [5],       [L],       50, 0.82, 240),
    ("SPT-EPL",  "Premier League Mornings",            "Live Sports",    "Soccer",        "Premier League", [8,9,10,11,12,1,2,3,4,5], [5,6],   [M],       25, 0.42, 120),
    ("SPT-MLS",  "MLS Matchday",                       "Live Sports",    "Soccer",        "MLS",            [2,3,4,5,6,7,8,9,10,11], [2,5,6],  [E],       25, 0.40, 120),
    ("SPT-WNBA", "WNBA Shared Reality",                "Live Sports",    "Basketball",    "WNBA",           [5,6,7,8,9,10],         WEEK,      [E],       30, 0.52, 130),
    ("SPT-F1",   "F1 Race Weekend",                    "Live Sports",    "Motorsport",    "Formula 1",      [3,4,5,6,7,8,9,10,11,12], [6],     [M, A],    40, 0.60, 150),
    ("SPT-MM",   "March Madness Dome",                 "Live Sports",    "Basketball",    "NCAA",           [3,4],                  WEEK,      [A, E],    40, 0.80, 150),
    ("FLM-AUR",  "Aurora: Chasing the Northern Lights","Immersive Film", "Nature",        "Cosm Studios",   ALL, WEEK, [M, A], 24, 0.35, 45),
    ("FLM-OCN",  "Ocean Odyssey 360",                  "Immersive Film", "Nature",        "Cosm Studios",   ALL, WEEK, [M, A], 24, 0.38, 50),
    ("FLM-UNI",  "Journey to the Edge of the Universe","Immersive Film", "Space",         "Cosm Studios",   ALL, WEEK, [M, A], 26, 0.42, 55),
    ("FLM-SAV",  "Wild Savanna",                       "Immersive Film", "Nature",        "Partner Studio", ALL, WEEK, [M, A], 22, 0.30, 40),
    ("FLM-ART",  "Dreamscape: Living Art",             "Immersive Film", "Art",           "Partner Studio", ALL, WEEK, [M, A], 25, 0.33, 45),
    ("FLM-DIN",  "Age of Giants: Dinosaurs",           "Immersive Film", "Family",        "Cosm Studios",   ALL, WEEK, [M, A], 22, 0.40, 40),
    ("CON-SYN",  "Synthwave Nights",                   "Concert",        "Electronic",    "Cosm Live",      ALL, [4,5],   [E, L], 38, 0.50, 90),
    ("CON-LSR",  "Laser Rock Classics",                "Concert",        "Rock",          "Cosm Live",      ALL, [4,5,6], [E, L], 35, 0.48, 90),
    ("CON-HIP",  "Hip-Hop Legends Immersive",          "Concert",        "Hip-Hop",       "Cosm Live",      ALL, [4,5],   [L],    42, 0.55, 100),
    ("CON-SYM",  "Symphony Under the Dome",            "Concert",        "Classical",     "Cosm Live",      ALL, [6],     [E],    48, 0.40, 110),
    ("SPC-COM",  "Comedy Under the Dome",              "Special Event",  "Comedy",        "Cosm Live",      ALL, [3,4,5], [L],    32, 0.45, 90),
    ("SPC-TRV",  "Dome Trivia Night",                  "Special Event",  "Trivia",        "Cosm Live",      ALL, [1,2],   [E],    15, 0.30, 120),
    ("SPC-KID",  "Kids Space Camp",                    "Special Event",  "Family",        "Cosm Studios",   ALL, [5,6],   [M],    20, 0.36, 60),
    ("SPC-PLN",  "Planetarium Live: Night Sky Tour",   "Special Event",  "Education",     "Cosm Studios",   ALL, [0,1,2,3], [E],  18, 0.28, 60),
]
COLS = ["content_id","title","category","genre","partner","months","weekdays","slots","base_price","popularity","runtime_min"]
CONTENT = [dict(zip(COLS, r)) for r in CONTENT_SPEC]
FORMAT = {"Live Sports": "Shared Reality Live", "Immersive Film": "Dome Original Film", "Concert": "Immersive Concert", "Special Event": "Live Hosted Event"}
RATING = {"Live Sports": "All Ages", "Immersive Film": "All Ages", "Concert": "18+", "Special Event": "All Ages"}
DAYS = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]

content_pdf = pd.DataFrame([dict(
    content_id=c["content_id"], title=c["title"], category=c["category"], genre=c["genre"],
    rights_partner=c["partner"], presentation_format=FORMAT[c["category"]],
    age_rating="21+" if c["content_id"] == "SPC-COM" else RATING[c["category"]],
    season_months=",".join(map(str, sorted(c["months"]))), available_days=",".join(DAYS[d] for d in c["weekdays"]),
    base_ticket_price=float(c["base_price"]), popularity_index=c["popularity"], runtime_minutes=c["runtime_min"],
    is_active=True) for c in CONTENT])

save(content_pdf, "content", "content_catalog", "Cosm content catalog: live sports programs, dome films, immersive concerts and special events with base pricing")

In [0]:
TEAMS = {
    "SPT-NBA":  ["Lakers","Clippers","Mavericks","Celtics","Warriors","Nuggets","Suns","Knicks","Heat","Bucks","Thunder","Spurs"],
    "SPT-NFL":  ["Rams","Chargers","Cowboys","Eagles","49ers","Chiefs","Packers","Bills","Ravens","Lions","Texans","Raiders"],
    "SPT-CFB":  ["USC","UCLA","Texas","Oklahoma","TCU","SMU","Texas A&M","Alabama","Georgia","Ohio State","Michigan","Oregon"],
    "SPT-NHL":  ["Kings","Ducks","Stars","Golden Knights","Oilers","Avalanche","Rangers","Bruins","Panthers","Maple Leafs"],
    "SPT-MLB":  ["Dodgers","Angels","Rangers","Astros","Padres","Giants","Yankees","Red Sox","Cubs","Braves"],
    "SPT-MLS":  ["LAFC","LA Galaxy","FC Dallas","Austin FC","Seattle Sounders","Inter Miami","Atlanta United","Houston Dynamo"],
    "SPT-WNBA": ["Sparks","Wings","Aces","Liberty","Fever","Sky","Mercury","Storm"],
    "SPT-EPL":  ["Arsenal","Liverpool","Manchester City","Chelsea","Tottenham","Manchester United","Newcastle","Aston Villa"],
    "SPT-MM":   ["Duke","UConn","Gonzaga","Kansas","Houston","Baylor","Arizona","Kentucky","North Carolina","Purdue"],
}
GRAND_PRIX = ["Bahrain","Saudi Arabian","Australian","Japanese","Miami","Monaco","Canadian","British","Italian","Singapore","United States","Mexico City","Las Vegas","Abu Dhabi"]
SLOT_TIMES = {M: (11, 0), A: (14, 30), E: (19, 0), L: (21, 30)}
DOW_F  = {0: .80, 1: .75, 2: .80, 3: .90, 4: 1.15, 5: 1.30, 6: 1.15}
SLOT_F = {M: .70, A: .85, E: 1.10, L: 1.00}
MARQUEE = {("SPT-NBA", 5): 1.3, ("SPT-NBA", 6): 1.4, ("SPT-NFL", 1): 1.35, ("SPT-NFL", 2): 1.5,
           ("SPT-CFB", 1): 1.25, ("SPT-MM", 3): 1.15, ("SPT-MM", 4): 1.3}
VENUE_F = {"COSM-LA": 1.0, "COSM-DAL": 0.92}
FILMS = [c for c in CONTENT if c["category"] == "Immersive Film"]

def event_title(c, d):
    cid = c["content_id"]
    if cid in TEAMS:
        a, b = rng.choice(TEAMS[cid], 2, replace=False)
        return f"{c['partner']}: {a} vs {b}"
    if cid == "SPT-UFC": return f"UFC Fight Night: {d:%b %d, %Y}"
    if cid == "SPT-F1":  return f"F1 {rng.choice(GRAND_PRIX)} Grand Prix"
    return c["title"]

rows = []
for v in venues_pdf[venues_pdf.status == "Open"].itertuples():
    open_d = pd.Timestamp(v.open_date)
    for d in pd.date_range(open_d, SCHEDULE_END, freq="D"):
        wd, age = d.weekday(), (d - open_d).days
        slots = [M, A, E] + ([L] if wd in (4, 5, 6) else [])
        for slot in slots:
            elig = [c for c in CONTENT if d.month in c["months"] and wd in c["weekdays"] and slot in c["slots"]] or FILMS
            w = np.array([c["popularity"] for c in elig]); c = elig[rng.choice(len(elig), p=w / w.sum())]
            start = d + pd.Timedelta(hours=SLOT_TIMES[slot][0], minutes=SLOT_TIMES[slot][1])
            on_sale = start - pd.Timedelta(days=14 if c["category"] == "Live Sports" else 45)
            occ = (c["popularity"] * DOW_F[wd] * SLOT_F[slot] * VENUE_F[v.venue_id]
                   * (1.25 if age < 90 else 1.0) * min(1 + 0.15 * age / 365, 1.3)
                   * MARQUEE.get((c["content_id"], d.month), 1.0) * rng.lognormal(0, 0.18))
            occ = float(np.clip(occ, 0.05, 0.99))
            if start < NOW:
                status = "Cancelled" if rng.random() < 0.01 else "Completed"
                frac = 0.5 if status == "Cancelled" else 1.0
            elif on_sale <= NOW:
                status, frac = "On Sale", float(np.clip(np.exp(-(start - NOW).days / 12), 0.08, 1))
            else:
                status, frac = "Announced", 0.0
            rows.append(dict(
                showtime_id=f"SHW-{v.venue_id[5:]}-{start:%Y%m%d%H%M}", venue_id=v.venue_id, content_id=c["content_id"],
                event_title=event_title(c, d), show_date=start.date(), time_slot=slot,
                start_time_local=start, end_time_local=start + pd.Timedelta(minutes=c["runtime_min"]),
                on_sale_ts=on_sale, status=status,
                # internal fields used for order generation (not persisted)
                _category=c["category"], _occ=occ, _base_price=c["base_price"], _runtime=c["runtime_min"],
                _tickets=int(round(v.seating_capacity * occ * frac))))

show_int = pd.DataFrame(rows)
showtimes_pdf = show_int[[c for c in show_int.columns if not c.startswith("_")]]
save(showtimes_pdf, "content", "showtimes", "Scheduled performances per venue (one row per show) with status Completed, Cancelled, On Sale or Announced")
display(showtimes_pdf.groupby(["venue_id", "status"]).size().reset_index(name="shows"))

In [0]:
FIRST = ["James","Maria","Michael","Jennifer","David","Linda","Carlos","Sofia","Daniel","Emily","Jose","Ashley","Kevin","Jessica","Brian","Sarah","Anthony","Nicole","Luis","Samantha","Jason","Olivia","Ryan","Isabella","Andrew","Mia","Eric","Ava","Marcus","Chloe","Tyler","Grace","Jordan","Hannah","Ethan","Priya","Wei","Aaliyah","Diego","Kayla"]
LAST = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis","Rodriguez","Martinez","Hernandez","Lopez","Gonzalez","Wilson","Anderson","Thomas","Taylor","Moore","Jackson","Martin","Lee","Perez","Thompson","White","Harris","Sanchez","Clark","Ramirez","Lewis","Robinson","Walker","Young","Allen","King","Wright","Nguyen","Kim","Patel","Chen","Rivera"]
MARKETS = {
    "COSM-LA":  dict(n=160_000, area=["310","323","424","213"], places=[("Inglewood","CA","90301"),("Los Angeles","CA","90045"),("Culver City","CA","90230"),("Santa Monica","CA","90401"),("Long Beach","CA","90802"),("Torrance","CA","90501"),("Pasadena","CA","91101"),("Hawthorne","CA","90250"),("Burbank","CA","91502"),("Anaheim","CA","92805")]),
    "COSM-DAL": dict(n=90_000,  area=["214","469","972","817"], places=[("The Colony","TX","75056"),("Frisco","TX","75034"),("Plano","TX","75024"),("Dallas","TX","75201"),("Lewisville","TX","75067"),("Carrollton","TX","75007"),("McKinney","TX","75070"),("Denton","TX","76201"),("Fort Worth","TX","76102"),("Irving","TX","75039")]),
}
parts = []
for vid, m in MARKETS.items():
    n = m["n"]; pl = np.array(m["places"])[rng.integers(0, len(m["places"]), n)]
    parts.append(pd.DataFrame(dict(
        first_name=rng.choice(FIRST, n), last_name=rng.choice(LAST, n),
        phone=[f"({a}) 555-{x:04d}" for a, x in zip(rng.choice(m["area"], n), rng.integers(0, 10000, n))],
        birth_year=(2026 - np.clip(rng.normal(34, 11, n), 18, 75)).astype(int),
        gender=rng.choice(["Female","Male","Non-binary","Prefer not to say"], n, p=[.46,.48,.02,.04]),
        city=pl[:, 0], state=pl[:, 1], postal_code=pl[:, 2], home_venue_id=vid,
        acquisition_channel=rng.choice(["Organic Search","Paid Social","Email Referral","Sports Partner","Friend Referral","Walk-in","Influencer"], n, p=[.22,.24,.10,.14,.14,.10,.06]),
        marketing_opt_in=rng.random(n) < 0.62)))
cust_base = pd.concat(parts, ignore_index=True)
cust_base.insert(0, "customer_id", [f"CUST-{i:07d}" for i in range(1, len(cust_base) + 1)])
cust_base.insert(3, "email", (cust_base.first_name + "." + cust_base.last_name + cust_base.index.astype(str) + "@"
                              + rng.choice(["example.com","mail.example.net","inbox.example.org"], len(cust_base))).str.lower())
print(f"{len(cust_base):,} customer profiles generated")

In [0]:
THEMES = ["Season Opener","Playoff Push","Summer Dome Nights","Holiday Gift Tickets","Back to School","Fight Night Bundle",
          "Family Weekend","Date Night","Membership Drive","Spring Break","New Film Premiere","Game Day Group Pack"]
camps, first_start, last_start = [], pd.Timestamp("2024-05-15"), pd.Timestamp("2026-12-01")
for i in range(60):
    s = first_start + pd.Timedelta(days=int(rng.integers(0, (last_start - first_start).days)))
    target = rng.choice(["ALL", "COSM-LA", "COSM-DAL"], p=[.4, .35, .25])
    if target == "COSM-DAL" and s < pd.Timestamp("2025-05-01"): target = "COSM-LA"
    theme, pct = rng.choice(THEMES), int(rng.choice([10, 15, 20, 25], p=[.35, .35, .2, .1]))
    camps.append(dict(
        campaign_id=f"CMP-{i + 1:03d}", campaign_name=f"{theme} {s.year}", theme=theme,
        channel=rng.choice(["Email","Paid Social","Paid Search","Influencer","Sports Partner","Out-of-Home"], p=[.25,.25,.15,.1,.15,.1]),
        objective=rng.choice(["Awareness","Acquisition","Retention","Upsell"]), target_venue_id=target,
        start_date=s.date(), end_date=(s + pd.Timedelta(days=int(rng.integers(14, 46)))).date(),
        budget_usd=float(rng.integers(5, 80) * 1000), promo_code="".join(w[0] for w in theme.split()).upper() + f"{pct}{i + 1:02d}",
        discount_pct=pct / 100))
campaigns_pdf = pd.DataFrame(camps).sort_values("start_date").reset_index(drop=True)

# One active promo per venue per day (used to apply promo codes to orders)
cal = []
for c in campaigns_pdf.itertuples():
    vids = ["COSM-LA", "COSM-DAL"] if c.target_venue_id == "ALL" else [c.target_venue_id]
    for d in pd.date_range(c.start_date, c.end_date, freq="D"):
        for vid in vids:
            cal.append((vid, d, c.campaign_id, c.promo_code, c.discount_pct))
camp_cal = (pd.DataFrame(cal, columns=["venue_id", "purchase_date", "campaign_id", "promo_code", "promo_pct"])
            .sample(frac=1, random_state=SEED).drop_duplicates(["venue_id", "purchase_date"]))

save(campaigns_pdf, "marketing", "campaigns", "Marketing campaigns with channel, budget, target venue and associated promo code")

In [0]:
st = show_int[show_int._tickets > 0].reset_index(drop=True)
idx = np.repeat(np.arange(len(st)), np.round(st._tickets.values / 2.6).astype(int))
o = st.iloc[idx][["showtime_id","venue_id","start_time_local","on_sale_ts","status","_category","_occ","_base_price","_runtime"]].reset_index(drop=True)
o = o.rename(columns={"status": "showtime_status"})
n = len(o)

o["quantity"] = rng.choice([1, 2, 3, 4, 5, 6], n, p=[.18, .42, .15, .15, .05, .05])
# Hard cap: tickets per showtime never exceed venue seating capacity
cap = o.venue_id.map(venues_pdf.set_index("venue_id").seating_capacity)
o = o[(o.groupby("showtime_id").quantity.cumsum() <= cap).values].reset_index(drop=True)
n = len(o)
zs = np.array([z[2] for z in ZONES])
o["zone_code"] = rng.choice([z[0] for z in ZONES], n, p=zs / zs.sum())
o["zone_id"] = o.venue_id + "-" + o.zone_code
o["channel"] = rng.choice(["Web", "Mobile App", "Box Office", "Partner Marketplace"], n, p=[.52, .31, .11, .06])
o.loc[(o.start_time_local >= NOW) & (o.channel == "Box Office"), "channel"] = "Web"
o["payment_method"] = rng.choice(["Credit Card", "Apple Pay", "Google Pay", "Debit Card", "Gift Card"], n, p=[.45, .22, .12, .16, .05])

# Purchase timing: sports bought closer to the event; box office same day; never before on-sale or after NOW
window_h = (o.start_time_local - o.on_sale_ts).dt.total_seconds().values / 3600
lead_h = rng.exponential(np.where(o._category == "Live Sports", 3.0, 8.0) * 24)
lead_h = np.clip(np.minimum(lead_h, window_h * rng.uniform(0.85, 1.0, n)), 0.25, None)
lead_h = np.where(o.channel == "Box Office", rng.uniform(0.1, 2.0, n), lead_h)
o["purchase_ts"] = o.start_time_local - pd.to_timedelta(lead_h, unit="h")
late = (o.purchase_ts >= NOW).values
span_s = (NOW - o.on_sale_ts[late]).dt.total_seconds().values
o.loc[late, "purchase_ts"] = (o.on_sale_ts[late] + pd.to_timedelta(rng.random(late.sum()) * span_s, unit="s")).values
o["purchase_ts"] = o.purchase_ts.dt.floor("s")

# Customer assignment: 88% home-market customers with heavy-tailed repeat frequency; 45% of box office are guests
pools = {}
for vid in MARKETS:
    ids = cust_base.customer_id.values[cust_base.home_venue_id.values == vid]
    w = rng.lognormal(0, 1.3, len(ids)); pools[vid] = (ids, w / w.sum())
cust_col = np.empty(n, dtype=object)
for vid in MARKETS:
    m = (o.venue_id == vid).values; k = m.sum()
    other = [x for x in MARKETS if x != vid][0]
    home = rng.choice(pools[vid][0], k, p=pools[vid][1]); away = rng.choice(pools[other][0], k, p=pools[other][1])
    cust_col[m] = np.where(rng.random(k) < 0.88, home, away)
cust_col[((o.channel == "Box Office").values) & (rng.random(n) < 0.45)] = None
o["customer_id"] = cust_col

o = o.sort_values("purchase_ts").reset_index(drop=True)
o.insert(0, "order_id", [f"ORD-{i:08d}" for i in range(1, n + 1)])
print(f"{n:,} orders, {o.quantity.sum():,} tickets")

In [0]:
# --- Cosm Club memberships: repeat buyers convert, starting after their 2nd order ---
og = o[o.customer_id.notna()].sort_values("purchase_ts")
cnt = og.groupby("customer_id").size()
second = og[og.groupby("customer_id").cumcount().values == 1].set_index("customer_id").purchase_ts
cand = cnt[cnt >= 2].index
mem_c = cand[rng.random(len(cand)) < np.minimum(0.55, 0.06 * cnt.loc[cand].values)]
mem = pd.DataFrame({"customer_id": mem_c, "start_ts": (second.loc[mem_c] + pd.to_timedelta(rng.uniform(0, 20, len(mem_c)), unit="D")).dt.floor("s").values})
mem = mem[mem.start_ts < NOW].reset_index(drop=True)
k = len(mem)
mem["tier"] = rng.choice(["Explorer", "Insider", "All-Access"], k, p=[.55, .35, .10])
mem["monthly_fee"] = mem.tier.map({"Explorer": 14.99, "Insider": 29.99, "All-Access": 59.99})
mem["member_discount_pct"] = mem.tier.map({"Explorer": .10, "Insider": .15, "All-Access": .20})
end = mem.start_ts + pd.to_timedelta(30 + rng.exponential(150, k), unit="D")
mem["end_ts"] = end.where((rng.random(k) < 0.30) & (end < NOW)).dt.floor("s")
mem["status"] = np.where(mem.end_ts.notna(), "Cancelled", "Active")
mem["auto_renew"] = mem.status.eq("Active") & (rng.random(k) < 0.9)
mem.insert(0, "membership_id", [f"MEM-{i:06d}" for i in range(1, k + 1)])

# --- Discounts: active member discount, or a campaign promo code if better ---
o["purchase_date"] = o.purchase_ts.dt.normalize()
o = o.merge(mem[["customer_id", "start_ts", "end_ts", "member_discount_pct"]], on="customer_id", how="left")
active = (o.purchase_ts >= o.start_ts) & (o.end_ts.isna() | (o.purchase_ts < o.end_ts))
member_pct = np.where(active, o.member_discount_pct.fillna(0), 0.0)
o = o.merge(camp_cal, on=["venue_id", "purchase_date"], how="left")
use_promo = o.campaign_id.notna().values & (rng.random(len(o)) < 0.12) & (o.promo_pct.fillna(0).values > member_pct)
o["discount_pct"] = np.where(use_promo, o.promo_pct.fillna(0), member_pct)
o["promo_code"] = o.promo_code.where(use_promo)
o["campaign_id"] = o.campaign_id.where(use_promo)
o["member_discount_applied"] = (member_pct > 0) & ~use_promo

# --- Pricing: base price x zone multiplier x demand-based dynamic pricing ---
mult = o.zone_code.map({z[0]: z[3] for z in ZONES})
o["unit_price"] = (np.round(o._base_price * mult * (0.85 + 0.4 * o._occ)) - 0.01).clip(lower=9.99)
o["subtotal"] = (o.unit_price * o.quantity).round(2)
o["discount_amount"] = (o.subtotal * o.discount_pct).round(2)
o["fees"] = (o.quantity * (o.unit_price * 0.12).round(2)).round(2)
TAX = {"COSM-LA": 0.095, "COSM-DAL": 0.0825}
o["tax"] = ((o.subtotal - o.discount_amount) * o.venue_id.map(TAX)).round(2)
o["total_amount"] = (o.subtotal - o.discount_amount + o.fees + o.tax).round(2)

# --- Status and attendance ---
past = (o.start_time_local < NOW).values
refund = (o.showtime_status == "Cancelled").values | (rng.random(len(o)) < 0.015)
o["order_status"] = np.select([refund, past], ["Refunded", "Completed"], "Confirmed")
o["tickets_scanned"] = np.where(o.order_status == "Completed", rng.binomial(o.quantity, 0.94), 0)

ORDER_COLS = ["order_id","customer_id","showtime_id","venue_id","zone_id","purchase_ts","channel","payment_method","quantity",
              "unit_price","subtotal","discount_pct","discount_amount","promo_code","campaign_id","member_discount_applied",
              "fees","tax","total_amount","order_status","tickets_scanned"]

# --- Finalize customers: signup before first order, rewards tier from lifetime spend ---
cust = cust_base.set_index("customer_id")
signup = pd.Series(LAUNCH + pd.to_timedelta(rng.random(len(cust)) * (NOW - LAUNCH).total_seconds(), unit="s"), index=cust.index)
first_order = og.groupby("customer_id").purchase_ts.min()
signup.loc[first_order.index] = (first_order - pd.to_timedelta(rng.exponential(20, len(first_order)), unit="D")).clip(lower=LAUNCH)
cust["signup_ts"] = signup.dt.floor("s")
spend = o[o.order_status == "Completed"].groupby("customer_id").total_amount.sum().reindex(cust.index).fillna(0)
cust["rewards_tier"] = pd.cut(spend, [-1, 200, 600, 1500, np.inf], labels=["Nova", "Star", "Galaxy", "Supernova"]).astype(str)
cust["is_club_member"] = cust.index.isin(mem.customer_id[mem.status == "Active"])
customers_pdf = cust.reset_index()

save(customers_pdf, "customers", "customers", "Customer CRM profiles with home venue, acquisition channel, marketing opt-in and Cosm Rewards tier")
save(mem.drop(columns=["member_discount_pct"]).assign(member_discount_pct=mem.member_discount_pct),
     "customers", "memberships", "Cosm Club paid memberships (Explorer, Insider, All-Access) with start/end and ticket discount")
save(o[ORDER_COLS], "sales", "ticket_orders", "Ticket orders: one row per order with zone, quantity, pricing, discounts, status and tickets scanned at entry")

In [0]:
MENU = [("FB01","Dome Lager Draft","Beer",11.0),("FB02","Local IPA","Beer",13.0),("FB03","Hard Seltzer","Beer",10.0),
        ("FB04","Cosmic Margarita","Cocktail",16.0),("FB05","Supernova Old Fashioned","Cocktail",18.0),("FB06","Galaxy Spritz","Cocktail",15.0),
        ("FB07","House Red Wine","Wine",14.0),("FB08","Sparkling Rose","Wine",15.0),
        ("FB09","Craft Soda","Non-Alcoholic",6.0),("FB10","Bottled Water","Non-Alcoholic",5.0),("FB11","Zero-Proof Mule","Non-Alcoholic",9.0),
        ("FB12","Loaded Nachos","Shareable",17.0),("FB13","Wings Platter","Shareable",19.0),("FB14","Truffle Fries","Shareable",12.0),
        ("FB15","Soft Pretzel and Beer Cheese","Shareable",13.0),("FB16","Smash Burger","Entree",18.0),("FB17","Chicken Tenders","Entree",15.0),
        ("FB18","Margherita Flatbread","Entree",16.0),("FB19","Gourmet Popcorn","Snack",8.0),("FB20","Churro Bites","Dessert",10.0),
        ("FB21","Ice Cream Sundae","Dessert",11.0)]
menu_pdf = pd.DataFrame(MENU, columns=["item_id", "item_name", "item_category", "price"])
menu_pdf["is_alcoholic"] = menu_pdf.item_category.isin(["Beer", "Cocktail", "Wine"])
W_SPORT = np.array([12,9,5,4,3,3,2,2,4,4,2,8,8,6,6,5,4,3,4,2,1], float); W_OTHER = np.array([5,4,4,6,4,5,5,5,6,6,3,5,3,5,3,3,4,4,10,6,5], float)

# Transactions only for attended orders; sports and concerts buy more
att = o[o.tickets_scanned > 0]
p_buy = np.select([att._category == "Live Sports", att._category == "Concert"], [0.72, 0.60], 0.40)
b = att[rng.random(len(att)) < p_buy].reset_index(drop=True)
nt = len(b)
txn = pd.DataFrame({
    "transaction_id": [f"FNB-{i:08d}" for i in range(1, nt + 1)], "order_id": b.order_id, "customer_id": b.customer_id,
    "venue_id": b.venue_id, "showtime_id": b.showtime_id,
    "transaction_ts": (b.start_time_local + pd.to_timedelta(rng.uniform(-45, 1, nt) * 1 + rng.uniform(0, 1, nt) * b._runtime.values, unit="m")).dt.floor("s"),
    "outlet": np.where(b.zone_code.isin(["BTH", "STE"]), "In-Seat Service", rng.choice(["Dome Bar", "Concourse Kitchen", "Lounge"], nt, p=[.45, .4, .15])),
    "payment_method": rng.choice(["Credit Card", "Apple Pay", "Google Pay", "Debit Card"], nt, p=[.5, .25, .1, .15])})

n_lines = 1 + rng.poisson(1.2 * np.sqrt(b.tickets_scanned.values))
rep = np.repeat(np.arange(nt), n_lines)
is_sport = (b._category.values == "Live Sports")[rep]
item_ix = np.where(is_sport, rng.choice(len(MENU), len(rep), p=W_SPORT / W_SPORT.sum()), rng.choice(len(MENU), len(rep), p=W_OTHER / W_OTHER.sum()))
lines = pd.DataFrame({"transaction_id": txn.transaction_id.values[rep], "item_id": menu_pdf.item_id.values[item_ix],
                      "quantity": 1 + rng.poisson(0.25, len(rep)), "unit_price": menu_pdf.price.values[item_ix]})
lines.insert(1, "line_number", lines.groupby("transaction_id").cumcount() + 1)
lines["line_total"] = (lines.quantity * lines.unit_price).round(2)

txn["subtotal"] = lines.groupby("transaction_id").line_total.sum().reindex(txn.transaction_id).values
txn["tax"] = (txn.subtotal * txn.venue_id.map(TAX)).round(2)
txn["tip"] = (txn.subtotal * np.where(txn.outlet.isin(["In-Seat Service", "Dome Bar"]), rng.uniform(0.15, 0.22, nt), 0)).round(2)
txn["total_amount"] = (txn.subtotal + txn.tax + txn.tip).round(2)

save(menu_pdf, "sales", "fnb_menu_items", "In-venue food and beverage menu items with category and list price")
save(txn, "sales", "fnb_transactions", "Food and beverage POS transactions (header) linked to the ticket order and showtime")
save(lines, "sales", "fnb_transaction_items", "Food and beverage POS line items per transaction")

In [0]:
s = o[(o.tickets_scanned > 0) & o.customer_id.notna()].sample(frac=0.07, random_state=SEED).reset_index(drop=True)
ns = len(s)
base = s._category.map({"Live Sports": 8.4, "Concert": 8.1, "Immersive Film": 7.9, "Special Event": 7.6}).values
raw = (base + 0.6 * (s._occ.values > 0.85) - 0.8 * (s.zone_code.values == "UPR")
       + 0.5 * np.isin(s.zone_code.values, ["BTH", "STE"]) + rng.normal(0, 1.9, ns))
score = np.clip(np.round(raw), 0, 10).astype(int)
cat = np.select([score >= 9, score >= 7], ["Promoter", "Passive"], "Detractor")
COMMENTS = {
    "Promoter":  ["Felt like being courtside, unreal experience", "The dome visuals are incredible", "Best way to watch a game outside the stadium",
                  "Staff were amazing and the food was great", "Already planning our next visit"],
    "Passive":   ["Fun night but drinks were pricey", "Great screen, seating was a bit tight", "Good experience, lines at the bar were long",
                  "Enjoyed it, parking was confusing"],
    "Detractor": ["Audio was too loud for us", "Upper tier view was not worth the price", "Long wait for food and the order was wrong",
                  "Stream had a few glitches during the game", "Too crowded to enjoy it"]}
comment = np.array([rng.choice(COMMENTS[c]) for c in cat], dtype=object)
comment[rng.random(ns) < 0.6] = None
rate = lambda adj: np.clip(np.round(score / 2 + adj + rng.normal(0, 0.7, ns)), 1, 5).astype(int)
nps = pd.DataFrame({
    "response_id": [f"NPS-{i:07d}" for i in range(1, ns + 1)], "order_id": s.order_id, "customer_id": s.customer_id,
    "venue_id": s.venue_id, "showtime_id": s.showtime_id,
    "response_ts": (s.start_time_local + pd.to_timedelta(rng.uniform(6, 72, ns), unit="h")).dt.floor("s"),
    "nps_score": score, "nps_category": cat, "rating_visual": rate(0.4), "rating_audio": rate(0.0),
    "rating_fnb": rate(-0.4), "rating_staff": rate(0.2), "comment": comment})
nps = nps[nps.response_ts < NOW]
save(nps, "marketing", "nps_survey_responses", "Post-visit survey responses: NPS (0-10), 1-5 ratings for visuals, audio, F&B, staff, and free-text comments")

## Public data source ingested with Lakeflow Connect
Two free, public APIs add external context that drives attendance at dome venues:
* **Open-Meteo Historical Weather API** (archive-api.open-meteo.com, CC BY 4.0) - daily weather at each Cosm venue location
* **Nager.Date Public Holidays API** (date.nager.at) - US public holidays

Pattern: raw JSON files land in the UC volume `/Volumes/jr_febar_catalog/external_data/landing`, then the **Lakeflow Connect file connector (Auto Loader, `cloudFiles`)** incrementally ingests them into bronze Delta tables with schema inference, file-level lineage and exactly-once checkpoints. New files dropped later are picked up on the next run. A curated `weather_daily` table flattens the API arrays to one row per venue per day.

In [0]:
LANDING = f"/Volumes/{CATALOG}/external_data/landing"
os.makedirs(f"{LANDING}/weather", exist_ok=True); os.makedirs(f"{LANDING}/holidays", exist_ok=True)
DAILY = ["weather_code","temperature_2m_max","temperature_2m_min","precipitation_sum","rain_sum","snowfall_sum","wind_speed_10m_max","sunshine_duration"]
weather_end = (NOW - pd.Timedelta(days=6)).date()   # archive API lags ~5 days

for v in venues_pdf.itertuples():
    for yr in range(2024, NOW.year + 1):
        start, end = max(date(yr, 1, 1), date(2024, 6, 1)), min(date(yr, 12, 31), weather_end)
        path = f"{LANDING}/weather/{v.venue_id}_{yr}_{end:%Y%m%d}.json"
        if os.path.exists(path): continue
        r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=60, params=dict(
            latitude=v.latitude, longitude=v.longitude, start_date=str(start), end_date=str(end), daily=",".join(DAILY),
            timezone=v.timezone, temperature_unit="fahrenheit", precipitation_unit="inch", wind_speed_unit="mph"))
        r.raise_for_status()
        payload = r.json() | {"venue_id": v.venue_id, "source": "open-meteo-archive", "fetched_at": pd.Timestamp.now(tz="UTC").isoformat()}
        with open(path, "w") as f: json.dump(payload, f)
        print("landed", path)

for yr in range(2024, NOW.year + 2):
    path = f"{LANDING}/holidays/us_public_holidays_{yr}.json"
    if os.path.exists(path): continue
    r = requests.get(f"https://date.nager.at/api/v3/PublicHolidays/{yr}/US", timeout=60); r.raise_for_status()
    with open(path, "w") as f: json.dump(r.json(), f)
    print("landed", path)

In [0]:
CHK = f"{LANDING}/_checkpoints"

def autoload(src_dir, target):
    """Incrementally ingest new JSON files from the landing volume into a bronze Delta table."""
    fq = f"{CATALOG}.external_data.{target}"
    (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json").option("multiLine", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", f"{CHK}/{target}/schema")
        .load(f"{LANDING}/{src_dir}")
        .select("*", F.col("_metadata.file_path").alias("_source_file"), F.current_timestamp().alias("_ingested_at"))
        .writeStream.option("checkpointLocation", f"{CHK}/{target}/checkpoint")
        .trigger(availableNow=True).toTable(fq)
    ).awaitTermination()
    print(f"{fq}: {spark.table(fq).count():,} rows")

autoload("weather", "weather_daily_raw")
autoload("holidays", "us_public_holidays")

# Curated: one row per venue per day
WMO = {0:"Clear",1:"Mainly clear",2:"Partly cloudy",3:"Overcast",45:"Fog",48:"Fog",51:"Drizzle",53:"Drizzle",55:"Drizzle",
       61:"Rain",63:"Rain",65:"Heavy rain",66:"Freezing rain",67:"Freezing rain",71:"Snow",73:"Snow",75:"Heavy snow",77:"Snow grains",
       80:"Rain showers",81:"Rain showers",82:"Violent rain showers",85:"Snow showers",86:"Snow showers",95:"Thunderstorm",96:"Thunderstorm with hail",99:"Thunderstorm with hail"}
wmo_map = F.create_map(*[F.lit(x) for kv in WMO.items() for x in kv])
raw = spark.table(f"{CATALOG}.external_data.weather_daily_raw")
weather = (raw.select("venue_id", "daily", F.posexplode("daily.time").alias("i", "d"))
    .select("venue_id", F.to_date("d").alias("weather_date"),
            *[F.col("daily")[c][F.col("i")].alias(c) for c in DAILY])
    .withColumn("weather_description", wmo_map[F.col("weather_code").cast("int")])
    .withColumn("sunshine_hours", F.round(F.col("sunshine_duration") / 3600, 1)).drop("sunshine_duration")
    .withColumn("is_rain_day", F.col("precipitation_sum") >= 0.1)
    .dropDuplicates(["venue_id", "weather_date"]))
weather.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.external_data.weather_daily")
spark.sql(f"COMMENT ON TABLE {CATALOG}.external_data.weather_daily IS 'Daily weather per Cosm venue (F, inches, mph) from Open-Meteo, flattened from weather_daily_raw'")
spark.sql(f"COMMENT ON TABLE {CATALOG}.external_data.weather_daily_raw IS 'Bronze: raw Open-Meteo archive API responses ingested by Auto Loader'")
spark.sql(f"COMMENT ON TABLE {CATALOG}.external_data.us_public_holidays IS 'Bronze: US public holidays from Nager.Date API ingested by Auto Loader'")
display(spark.table(f"{CATALOG}.external_data.weather_daily").orderBy("venue_id", "weather_date").limit(10))

In [0]:
tables = spark.sql(f"SELECT table_schema, table_name FROM {CATALOG}.information_schema.tables WHERE table_schema <> 'information_schema' ORDER BY 1, 2").collect()
display(spark.createDataFrame([(r.table_schema, r.table_name, spark.table(f"{CATALOG}.{r.table_schema}.{r.table_name}").count()) for r in tables],
                              "schema string, table string, row_count long"))

checks = spark.sql(f"""
WITH occ AS (
  SELECT o.showtime_id, SUM(o.quantity) / MAX(v.seating_capacity) AS occupancy
  FROM {CATALOG}.sales.ticket_orders o JOIN {CATALOG}.venues.venues v USING (venue_id) GROUP BY o.showtime_id)
SELECT
  (SELECT COUNT(*) FROM {CATALOG}.sales.ticket_orders o LEFT ANTI JOIN {CATALOG}.content.showtimes s USING (showtime_id)) AS orphan_orders,
  (SELECT COUNT(*) FROM {CATALOG}.sales.ticket_orders o WHERE customer_id IS NOT NULL AND customer_id NOT IN (SELECT customer_id FROM {CATALOG}.customers.customers)) AS orphan_customers,
  (SELECT COUNT(*) FROM {CATALOG}.sales.ticket_orders o JOIN {CATALOG}.content.showtimes s USING (showtime_id) WHERE o.purchase_ts > s.start_time_local OR o.purchase_ts < s.on_sale_ts) AS bad_purchase_times,
  (SELECT COUNT(*) FROM {CATALOG}.sales.ticket_orders o JOIN {CATALOG}.customers.customers c USING (customer_id) WHERE c.signup_ts > o.purchase_ts) AS orders_before_signup,
  (SELECT ROUND(MAX(occupancy), 3) FROM occ) AS max_occupancy,
  (SELECT ROUND(AVG(occupancy), 3) FROM occ) AS avg_occupancy,
  (SELECT COUNT(*) FROM {CATALOG}.sales.fnb_transactions f LEFT ANTI JOIN {CATALOG}.sales.ticket_orders o USING (order_id)) AS orphan_fnb,
  (SELECT MAX(purchase_ts) FROM {CATALOG}.sales.ticket_orders) AS latest_purchase_ts
""")
display(checks)

In [0]:
# comment for re-commit